In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, Math 

**Kolmogorov's energy spectrum in the inertial subrange** (a steady-state regime in energy spectrum)

${
E(k) = E_0 \, 
            k^{-5/3}
}
$
where, 

${
E_0 = 1.5 \, 
            \epsilon^{2/3} 
            \left[ 
                1 - \left(\frac{\eta}{L}\right) ^{4/3}
            \right]^{-1}
}
$
with kinematic viscocity 
$
\eta = {\left( \frac{\nu^3}{\epsilon} \right)}^{1/4}
$

### 1. Setting up the Kolmogorov energy speectrum


In [ ]:
# Parameters (in SI units)

# Dissipation rate 
# Ranges from 1e-4 (rough sea) to 1e-14 (calm sea)
epsilon = 1e-4   

# Maximum length scale in meters
L = 10   

# Kolmogorov length scale in meters
# Has a dependence on dissipation rate and kinematic viscosity (nu)
# Kinematic viscosity of sea water depends on salinity and temperature
# Ranges from 1e-6 to 1.8e-6
nu = 1e-6
eta = (nu**3 / epsilon) ** (1/4)   

# Calculate E0 (amplitude of energy spectrum)
E0 = 1.5 * epsilon**(2/3) * (1 - (eta/L)**(4/3))**(-1) 

# ----
# Linear sampling of wavenumbers (not ideal)
# Calculate wavenumber range
k_min = 2 * np.pi / L
k_max = 2 * np.pi / eta

k = np.linspace(k_min, k_max, 500)

# Calculate energy spectrum
E = E0 * (k**(-5/3))

# # Plot
plt.figure(figsize=(10, 6))
plt.loglog(k, E, label=r'$E(k) = E_0 \cdot k^{-5/3}$')
plt.xlabel('Wave number $k$')
plt.ylabel('Energy spectrum $E(k)$')
plt.title("Kolmogorov Energy Spectrum in the Inertial Range")
plt.legend()
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()

**Sampling N wave number distributed in a geometric series from $k_1$ to $k_N$**

In [ ]:
# Total number of wave numbers sampled 
N = 50

# Generate wavevectors k_n using the geometric series in ascending order
k_values = (2 * np.pi / L) * (L / eta) ** (np.arange(N) / (N - 1))

# Energy spectrum
E_k = E0 * (k_values ** (-5 / 3))

# Delta k values
delta_k0 = (k_values[1] - k_values[0]) / 2
delta_kN = (k_values[-1] - k_values[-2]) / 2
delta_k_ = (k_values[2:] - k_values[:-2]) / 2
delta_k = np.concatenate(([delta_k0], delta_k_, [delta_kN]))

Plot the Kolmogorov energy specutrm in the intertial subrange

In [ ]:
plt.figure(figsize=(10, 6))
plt.loglog(k_values, E_k, marker='o', label=r'$E(k) = E_0 \cdot k^{-5/3}$', color="darkviolet")
plt.xlabel('Wave number $k$')
plt.ylabel('Energy spectrum $E(k)$')
plt.title("Kolmogorov Energy Spectrum in the Inertial Range")
plt.legend()
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()

### 2. **Kinematic simulation of velocity fields**

$$\mathbf{u}(\mathbf{x}, t) = \sum_{n=1}^{N} \left[ \mathbf{A}_n \cos(\mathbf{k}_n \cdot \mathbf{x} + \omega_n t) + \mathbf{B}_n \sin(\mathbf{k}_n \cdot \mathbf{x} + \omega_n t) \right]$$

In [ ]:
# Simulation parameters
grid_size = 50          # Size of the 2D grid (50x50 points)
timesteps = 1000        # Number of time steps to simulate

# Amplitudes of the Fourier modes
a_n = b_n = np.sqrt(2 * E_k * delta_k)

# Temporal frequencies
omega_n = 0.4 * np.sqrt((k_values ** 3) * E_k)

# Random phases for amplitudes and wavevectors
angles = 2 * np.pi * np.random.rand(N)

# Define A_n, B_n, and k_n according to the incompressibility constraints (Shape: (N, 2)), where N = number of Fourier modes
A_n = np.array([a_n * np.cos(angles), -a_n * np.sin(angles)]).T
B_n = np.array([-b_n * np.cos(angles), b_n * np.sin(angles)]).T
k_n = np.array([k_values * np.sin(angles), k_values * np.cos(angles)]).T

# Define the spatial grid (x, y)
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)  # Create a grid of points (coordinates) (Shape: (grid_size, grid_size))
positions = np.stack([X.ravel(), Y.ravel()], axis=-1)  # Flattened grid positions for efficiency (Shape: (grid_size * grid_size, 2))

# Initialize an array to store the velocity field at each time step
velocity_field = np.zeros((timesteps, grid_size, grid_size, 2))

# Compute the velocity field for each time step
for t in range(timesteps):
    # Initialize the velocity at each point to zero
    u = np.zeros((grid_size * grid_size, 2))

    # Loop over each Fourier mode
    for n in range(N):
        # Compute the phase shift for each mode
        phase = np.dot(positions, k_n[n]) + omega_n[n] * t

        # Reshape A_n[n] and B_n[n] to be (1, 2) for broadcasting (for multiplication to all points in the grid)
        A_n_n = A_n[n].reshape(1, 2)
        B_n_n = B_n[n].reshape(1, 2)

        # Add the contribution of this mode to the velocity field
        u += (A_n_n * np.cos(phase)[:, None] + B_n_n * np.sin(phase)[:, None])


    # Reshape and store the velocity field for this time step
    velocity_field[t, ..., 0] = u[:, 0].reshape(grid_size, grid_size)  # x-component
    velocity_field[t, ..., 1] = u[:, 1].reshape(grid_size, grid_size)  # y-component

# velocity_field now contains the velocity vectors at each point on the grid for each time step


Plot quiver plot of velocity field

In [ ]:
import matplotlib.pyplot as plt

# Select a time step
t = 0  # Choose a specific time step (e.g., t=0 for the initial field)

# Extract the velocity components at time t
U = velocity_field[t, ..., 0]  # x-component of the velocity
V = velocity_field[t, ..., 1]  # y-component of the velocity

# Create the grid for plotting
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Plot the velocity field using quiver
plt.figure(figsize=(8, 8))
plt.quiver(X, Y, U, V, scale=50, pivot='mid', color='blue')
plt.title(f"Velocity Field at Time Step {t}")
plt.xlabel("x")
plt.ylabel("y")
plt.grid()
plt.show()

# plt.figure(figsize=(8, 8))
# plt.streamplot(X, Y, U, V, density=1.5, color=np.sqrt(U**2 + V**2), cmap='viridis')
# plt.colorbar(label="Velocity Magnitude")
# plt.title(f"Streamline Plot of Velocity Field at Time Step {t}")
# plt.xlabel("x")
# plt.ylabel("y")
# plt.grid()
# plt.show()

Quiver plot of velocity fields: Video

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Set up the figure and axis for the quiver plot
fig, ax = plt.subplots(figsize=(8, 8))

# Create the grid for plotting
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Initialize the quiver plot with the first time step
quiver = ax.quiver(X, Y, velocity_field[0, ..., 0], velocity_field[0, ..., 1], scale=50, pivot='mid', color='blue')
ax.set_title("Velocity Field over Time")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid()

# Function to update the plot for each frame
def update(frame):
    # Use precomputed U and V components for the current frame
    U = velocity_field[frame, ..., 0]
    V = velocity_field[frame, ..., 1]
    quiver.set_UVC(U, V)  # Update the quiver plot with new velocities
    ax.set_title(f"Velocity Field at Time Step {frame}")
    return quiver,

# Create the animation
anim = FuncAnimation(fig, update, frames=timesteps, interval=50, blit=True)
HTML(anim.to_jshtml())


Verifying the interpolation

**Note**: Scipy grid interpolator expects "physical" coordinates as inputs, not indices.

In [ ]:
from scipy.interpolate import RegularGridInterpolator

# Create a regular grid interpolator for the velocity field
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)

# Select a timestep for testing
timestep = 0
u_interpolator = RegularGridInterpolator((x, y), velocity_field[timestep, ..., 0])  # x-component of velocity
v_interpolator = RegularGridInterpolator((x, y), velocity_field[timestep, ..., 1])  # y-component of velocity

# Check interpolation at a specific grid point by using coordinates, not indices
index = 3  # Example grid index (4th row and 4th column in zero-based index)
point = [x[index], y[index]]  # Use actual physical coordinates

# Interpolated velocity at this physical point
u_point = u_interpolator(point)
v_point = v_interpolator(point)
print(f"Interpolated velocity at point {point}: ({u_point}, {v_point})")

# Compare with the actual velocity at the closest grid point using indices
u_grid = velocity_field[timestep, index, index, 0]
v_grid = velocity_field[timestep, index, index, 1]
print(f"Velocity at grid index [{index}, {index}]: ({u_grid}, {v_grid})")


### 3. Coupling Planktons to turbulence field (Stocastic differential equations)

Main physics of the entire simulation, coupling the velocity field with the plankton swimming and random noise

In [ ]:
### Set Plankton parameters
set_turbulence = True
n_planktons = 100
# velocity = 0.2
velocity = 0.3
Dt = 0.0001
delta_t = 0.1
angle = 10 # in degrees (plus or minus around the target angle 90 degrees)

target_angle = np.pi / 2
response_angle = angle * (np.pi / 180)

# Define velocity interpolators
x_grid = np.linspace(0, L, grid_size)
y_grid = np.linspace(0, L, grid_size)

def get_velocity_interpolators(time_step):
    vx_interpolator = RegularGridInterpolator((x_grid, y_grid), velocity_field[time_step, ..., 0])
    vy_interpolator = RegularGridInterpolator((x_grid, y_grid), velocity_field[time_step, ..., 1])
    return vx_interpolator, vy_interpolator

# Store the positions
stored_positions = np.zeros((n_planktons, 2, timesteps))

# Intialize the plankton positions and orientations
# x_pos = np.random.rand(n_planktons) * L
# y_pos = np.random.rand(n_planktons) * L
x_pos = np.random.uniform(0, L, n_planktons)
y_pos = np.random.uniform(0, L, n_planktons)
phi = np.random.rand(n_planktons) * 2 * np.pi

for t in range(timesteps):

    phi = target_angle + response_angle * (2 * np.random.rand(n_planktons) - 1)

    # turbulence velocities
    vx_interpolator, vy_interpolator = get_velocity_interpolators(t)
    positions = np.stack([x_pos, y_pos], axis=-1)
    vx_turb = vx_interpolator(positions)
    vy_turb = vy_interpolator(positions)

    if set_turbulence is False:
        vx_turb = 0
        vy_turb = 0

    # behavioral velocities
    vx_behav = velocity * np.cos(phi)
    vy_behav = velocity * np.sin(phi)

    # update positions
    x_pos = x_pos + (vx_turb + vx_behav) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons)
    y_pos = y_pos + (vy_turb + vy_behav) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons)

    # reflect at the boundaries
    x_pos[x_pos > L] = 2 * L - x_pos[x_pos > L]
    x_pos[x_pos < 0] = -x_pos[x_pos < 0]
    y_pos[y_pos > L] = 2 * L - y_pos[y_pos > L]
    y_pos[y_pos < 0] = -y_pos[y_pos < 0]

    # store the positions
    stored_positions[:, 0, t] = x_pos
    stored_positions[:, 1, t] = y_pos

**Plot plankter positions and timepoint**

In [ ]:
# t = 0 
# x_ = stored_positions[:, 0, t]
# y_ = stored_positions[:, 1, t]

# fig, ax = plt.subplots(figsize=(5, 5))
# scat = ax.scatter(x_, y_, c='white', s=20, alpha=0.5)
# # trails = [ax.plot([], [], '-', linewidth=1, alpha=0.5)[0] for _ in range(n_planktons)]
# ax.set_xlim(0, L)
# ax.set_ylim(0, L)
# ax.set_facecolor('black')


# def update(frame):
#     x_ = stored_positions[:, 0, frame]
#     y_ = stored_positions[:, 1, frame]
#     scat.set_offsets(np.c_[x_, y_])
#     # for i in range(n_planktons):
#     #     trails[i].set_data(stored_positions[i, 0, :frame], stored_positions[i, 1, :frame])
#     return scat, # *trails

# anim = FuncAnimation(fig, update, frames=timesteps, interval=50, blit=True)
# HTML(anim.to_jshtml())

**Quiver velocity field with plankton swimming through it**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Set up the figure and axis for the quiver plot
fig, ax = plt.subplots(figsize=(8, 8))

# Create the grid for plotting
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Initialize the quiver plot with the first time step
quiver = ax.quiver(X, Y, velocity_field[0, ..., 0], velocity_field[0, ..., 1], scale=50, pivot='mid', color='blue', alpha=0.2)
ax.set_title("Velocity Field over Time")
ax.set_xlabel("x")
ax.set_ylabel("y")
# ax.grid()

# Add the plankton positions
scat = ax.scatter(stored_positions[:, 0, 0], stored_positions[:, 1, 0], c='black', s=30, alpha=0.5)
trails = [ax.plot([], [], '-', linewidth=1, color="black", alpha=0.5)[0] for _ in range(n_planktons)]
ax.set_xlim(0, L)
ax.set_ylim(0, L)
# ax.set_axis_off()

# Function to update the plot for each frame
def update(frame):
    # Use precomputed U and V components for the current frame
    U = velocity_field[frame, ..., 0]
    V = velocity_field[frame, ..., 1]
    quiver.set_UVC(U, V)  # Update the quiver plot with new velocities
    ax.set_title(f"Velocity Field at Time Step {frame}")
    scat.set_offsets(np.c_[stored_positions[:, 0, frame], stored_positions[:, 1, frame]])

    for i in range(n_planktons):
        trails[i].set_data(stored_positions[i, 0, :frame], stored_positions[i, 1, :frame])
    return quiver, scat

# # Function to update the plot for each frame
# def update(frame):
#     # Use precomputed U and V components for the current frame
#     U = velocity_field[frame, ..., 0]
#     V = velocity_field[frame, ..., 1]
#     quiver.set_UVC(U, V)  # Update the quiver plot with new velocities
#     ax.set_title(f"Velocity Field at Time Step {frame}")
#     return quiver,

# # Create the animation
anim = FuncAnimation(fig, update, frames=timesteps, interval=50, blit=True)
# HTML(anim.to_jshtml())

# save the animation as a gif
# anim.save('plankton_simulation.gif', writer='imagemagick', fps=20)

save_path = r"C:\\Users\\isaeh\\GitHub\\sea-surface-optics-guide-zooplankton-towards-low-turbulence\\simulations"

file_name = "simulation_moderate.gif"

anim.save(
    os.path.join(save_path, file_name),
    writer="imagemagick",
    fps=20
)


**Animated streamplot of the velocity field through all timesteps**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Set up the figure and axis for the streamline plot
fig, ax = plt.subplots(figsize=(8, 8))

# Create the grid for plotting
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Initialize the plot with the first time step (without calling streamplot here)
U = velocity_field[0, ..., 0]
V = velocity_field[0, ..., 1]
ax.set_title("Velocity Field over Time")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.grid()

# Function to update the plot for each frame
def update_stream(frame):
    # Clear the axis instead of trying to clear individual elements
    ax.clear()
    
    # Set the title and labels again after clearing
    ax.set_title(f"Velocity Field at Time Step {frame}")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid()

    # Get U and V components for the current frame
    U = velocity_field[frame, ..., 0]
    V = velocity_field[frame, ..., 1]
    
    # Redraw the streamplot for this frame
    stream = ax.streamplot(X, Y, U, V, density=1.5, color=np.sqrt(U**2 + V**2), cmap='viridis')
    return stream

# Create the animation
anim = FuncAnimation(fig, update_stream, frames=timesteps, interval=50)
HTML(anim.to_jshtml())


**Streamplot or quiver plot of the velocity field at different timesteps**

In [ ]:
import matplotlib.pyplot as plt

for t in range(0, 10):

    # Select a time step
    # t = 0  # Choose a specific time step (e.g., t=0 for the initial field)

    # Extract the velocity components at time t
    U = velocity_field[t, ..., 0]  # x-component of the velocity
    V = velocity_field[t, ..., 1]  # y-component of the velocity

    # Create the grid for plotting
    x = np.linspace(0, L, grid_size)
    y = np.linspace(0, L, grid_size)
    X, Y = np.meshgrid(x, y)

    # Plot the velocity field using quiver
    # plt.figure(figsize=(8, 8))
    # plt.quiver(X, Y, U, V, scale=50, pivot='mid', color='blue')
    # plt.title(f"Velocity Field at Time Step {t}")
    # plt.xlabel("x")
    # plt.ylabel("y")
    # plt.grid()
    # plt.show()

    # Plot the velocity field using streamplot
    plt.figure(figsize=(8, 8))
    plt.streamplot(X, Y, U, V, density=1.5, color=np.sqrt(U**2 + V**2), cmap='viridis')
    plt.colorbar(label="Velocity Magnitude")
    plt.title(f"Streamline Plot of Velocity Field at Time Step {t}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid()
    plt.show()
